# Versioning Prompts, Models, and Configs in Production

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/12-versioning-prompts-models-configs.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your AI service was working fine last Tuesday. Today it is producing subtly different outputs. Nothing in your git history changed. No deployments happened. What broke?

The model provider quietly updated the endpoint behind `claude-haiku-latest`. Or your config file got a one-line edit that changed the system prompt temperature. Or a colleague updated the prompt template while you were asleep, and the change went live without any record of when or by whom.

AI services have three moving parts that code versioning does not capture:

1. The prompt template (changes frequently, often by non-engineers)
2. The model identifier (can change under you when you use aliases)
3. The service config (te...

## The Concept

### Three Components, One Manifest

```
+-------------------+    +-------------------+    +-------------------+
|  PROMPT TEMPLATE  |    |    MODEL ID        |    |  SERVICE CONFIG   |
|                   |    |                    |    |                   |
|  version: v1.2    |    |  claude-3-5-haiku  |    |  hash: a4f9c2b1   |
|  commit: abc123   |    |  -20241022         |    |  temp: 0.3        |
|  author: alice    |    |  (pinned, not      |    |  max_tokens: 512  |
|                   |    |   an alias)        |    |  retries: 3       |
+-------------------+    +-------------------+    ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 12: Versioning Prompts, Models, and Configs in Production
Phase 06: Shipping

A VersionManifest ties together the three moving parts of a production AI service:
  - prompt_version: which prompt template is active
  - model_id: the pinned model identifier (never an alias)
  - config_hash: SHA-256 fingerprint of the service config dict

Usage:
    python main.py              # run the demo: register two versions, roll back
    uvicorn main:app --reload   # start the FastAPI service (requires manifests.yaml)
"""

from __future__ import annotations

import hashlib
import json
import logging
from contextlib import asynccontextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import anthropic
import yaml
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

### Core data types

In [ ]:
MANIFEST_FILE = Path("manifests.yaml")


@dataclass
class VersionManifest:
    """Records the exact combination of prompt, model, and config deployed together."""

    manifest_id: str   # e.g. "v1.2.0"
    prompt_version: str  # e.g. "v1.2"
    model_id: str      # e.g. "claude-3-5-haiku-20241022" - pinned, never alias
    config_hash: str   # first 8 hex chars of SHA-256 of the config dict
    deployed_at: str   # ISO-8601 UTC timestamp
    deployed_by: str = "local"
    notes: str = ""


def hash_config(config: dict) -> str:
    """
    Compute a short, stable SHA-256 hash of a config dict.
    Keys are sorted so insertion order does not affect the hash.
    Returns first 8 hex characters.
    """
    serialized = json.dumps(config, sort_keys=True, ensure_ascii=True)
    return hashlib.sha256(serialized.encode()).hexdigest()[:8]


def make_manifest(
    manifest_id: str,
    prompt_version: str,
    model_id: str,
    config: dict,
    deployed_by: str = "local",
    notes: str = "",
) -> VersionManifest:
    """Factory: builds a VersionManifest from raw inputs. Computes the config hash."""
    return VersionManifest(
        manifest_id=manifest_id,
        prompt_version=prompt_version,
        model_id=model_id,
        config_hash=hash_config(config),
        deployed_at=datetime.now(timezone.utc).isoformat(),
        deployed_by=deployed_by,
        notes=notes,
    )

### Registry

In [ ]:
class ManifestRegistry:
    """
    Loads, saves, and queries VersionManifest records from a YAML file.
    The YAML file is git-tracked alongside your code.
    """

    def __init__(self, path: Path = MANIFEST_FILE) -> None:
        self.path = path
        self._manifests: list[VersionManifest] = []
        self._current_id: Optional[str] = None
        if self.path.exists():
            self._load()

    def _load(self) -> None:
        data = yaml.safe_load(self.path.read_text())
        if not data:
            return
        self._current_id = data.get("current")
        for entry in data.get("history", []):
            self._manifests.append(VersionManifest(**entry))

    def _save(self) -> None:
        data = {
            "current": self._current_id,
            "history": [asdict(m) for m in self._manifests],
        }
        self.path.write_text(yaml.dump(data, default_flow_style=False, sort_keys=True))

    def register(self, manifest: VersionManifest) -> None:
        """Add a new manifest and mark it as current. Rejects model aliases."""
        model_lower = manifest.model_id.lower()
        if "latest" in model_lower or model_lower.endswith(("-turbo", "-preview")):
            raise ValueError(
                f"Model alias '{manifest.model_id}' is not allowed. "
                "Use a pinned model ID such as 'claude-3-5-haiku-20241022'."
            )
        self._manifests.append(manifest)
        self._current_id = manifest.manifest_id
        self._save()

    def current(self) -> Optional[VersionManifest]:
        """Return the currently active manifest."""
        if not self._current_id:
            return None
        return self.get(self._current_id)

    def get(self, manifest_id: str) -> Optional[VersionManifest]:
        """Retrieve a specific manifest by ID."""
        for m in self._manifests:
            if m.manifest_id == manifest_id:
                return m
        return None

    def history(self) -> list[VersionManifest]:
        """Return all manifests in registration order (oldest first)."""
        return list(self._manifests)

### Rollback

In [ ]:
def rollback(registry: ManifestRegistry, manifest_id: str) -> VersionManifest:
    """
    Roll back to a previous manifest by ID.

    This changes which manifest is marked as current. The caller must
    ensure the corresponding config is also restored - the manifest is
    an index, not a config backup.
    """
    target = registry.get(manifest_id)
    if target is None:
        available = [m.manifest_id for m in registry.history()]
        raise ValueError(
            f"Manifest '{manifest_id}' not found. Available: {available}"
        )
    registry._current_id = manifest_id
    registry._save()
    logger.info("Rolled back to manifest %s", manifest_id)
    logger.info("  prompt_version: %s", target.prompt_version)
    logger.info("  model_id:       %s", target.model_id)
    logger.info("  config_hash:    %s", target.config_hash)
    logger.info("  deployed_at:    %s", target.deployed_at)
    return target

### FastAPI integration

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    """Log the active manifest at startup. Fail fast if none is registered."""
    registry = ManifestRegistry(MANIFEST_FILE)
    manifest = registry.current()

    if manifest is None:
        raise RuntimeError(
            "No active manifest found. "
            "Run 'python main.py' first to register a manifest."
        )

    logger.info("=== SERVICE STARTUP ===")
    logger.info("manifest_id:     %s", manifest.manifest_id)
    logger.info("prompt_version:  %s", manifest.prompt_version)
    logger.info("model_id:        %s", manifest.model_id)
    logger.info("config_hash:     %s", manifest.config_hash)
    logger.info("deployed_at:     %s", manifest.deployed_at)
    logger.info("deployed_by:     %s", manifest.deployed_by)
    if manifest.notes:
        logger.info("notes:           %s", manifest.notes)
    logger.info("=== STARTUP COMPLETE ===")

    app.state.manifest = manifest
    app.state.registry = registry

    yield

    logger.info("Service shutting down.")


app = FastAPI(title="Versioned AI Service", lifespan=lifespan)
_client = anthropic.Anthropic()


class ChatRequest(BaseModel):
    message: str


@app.get("/health")
async def health():
    """Returns active manifest with every health check. First thing ops checks."""
    manifest = app.state.manifest
    return {
        "status": "ok",
        "manifest_id": manifest.manifest_id,
        "prompt_version": manifest.prompt_version,
        "model_id": manifest.model_id,
        "config_hash": manifest.config_hash,
        "deployed_at": manifest.deployed_at,
        "deployed_by": manifest.deployed_by,
    }


@app.post("/chat")
async def chat(request: ChatRequest):
    """
    Chat endpoint. Uses the model ID from the active manifest.
    Logs which manifest served each request for traceability.
    """
    manifest = app.state.manifest

    try:
        response = _client.messages.create(
            model=manifest.model_id,
            max_tokens=512,
            messages=[{"role": "user", "content": request.message}],
        )
    except anthropic.APIError as exc:
        raise HTTPException(status_code=502, detail=str(exc)) from exc

    return {
        "response": response.content[0].text,
        "manifest_id": manifest.manifest_id,
        "model_id": manifest.model_id,
        "prompt_tokens": response.usage.input_tokens,
        "completion_tokens": response.usage.output_tokens,
    }


@app.post("/rollback/{manifest_id}")
async def rollback_endpoint(manifest_id: str):
    """
    Roll back to a previous manifest by ID.
    In a real system this would require authentication.
    The service must restart to pick up the new active manifest.
    """
    registry = app.state.registry
    try:
        target = rollback(registry, manifest_id)
    except ValueError as exc:
        raise HTTPException(status_code=404, detail=str(exc)) from exc
    return {
        "rolled_back_to": target.manifest_id,
        "note": "Restart the service to activate the rolled-back manifest.",
    }


# ---------------------------------------------------------------------------
# Demo: run from command line to register manifests and test rollback
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import sys

demo_path = Path("demo_manifests.yaml")
registry = ManifestRegistry(demo_path)

config_v1 = {
    "temperature": 0.3,
    "max_tokens": 512,
    "retries": 3,
    "system_prompt": "You are a helpful assistant. Be concise.",
}
config_v2 = {
    "temperature": 0.7,
    "max_tokens": 1024,
    "retries": 3,
    "system_prompt": "You are a helpful assistant. Be detailed and thorough.",
}

print("=== Registering v1.0.0 ===")
m1 = make_manifest(
    manifest_id="v1.0.0",
    prompt_version="v1.0",
    model_id="claude-3-5-haiku-20241022",
    config=config_v1,
    deployed_by="alice",
    notes="Initial production deploy",
)
registry.register(m1)
print(f"Registered: {m1.manifest_id}  config_hash={m1.config_hash}")

print("\n=== Registering v1.1.0 ===")
m2 = make_manifest(
    manifest_id="v1.1.0",
    prompt_version="v1.1",
    model_id="claude-3-5-haiku-20241022",
    config=config_v2,
    deployed_by="bob",
    notes="Higher temperature for more creative responses",
)
registry.register(m2)
print(f"Registered: {m2.manifest_id}  config_hash={m2.config_hash}")

print(f"\nCurrent manifest: {registry.current().manifest_id}")

print("\n=== Testing alias rejection ===")
try:
    bad = make_manifest(
        manifest_id="v1.2.0",
        prompt_version="v1.2",
        model_id="claude-haiku-latest",  # alias - should be rejected
        config=config_v1,
    )
    registry.register(bad)
    print("ERROR: alias was not rejected!")
    sys.exit(1)
except ValueError as e:
    print(f"Correctly rejected alias: {e}")

print("\n=== Rolling back to v1.0.0 ===")
rollback(registry, "v1.0.0")
print(f"Active after rollback: {registry.current().manifest_id}")

print("\n=== Full History ===")
for m in registry.history():
    marker = " <-- current" if m.manifest_id == registry.current().manifest_id else ""
    print(
        f"  {m.manifest_id:<10}  prompt={m.prompt_version}  "
        f"model={m.model_id}  config={m.config_hash}{marker}"
    )

print(f"\nManifest file written to: {demo_path.absolute()}")
print("Inspect it with: cat demo_manifests.yaml")

---

## Self-Check

Answer these without running code first:

**1. What does this scenario illustrate about versioning AI services?**

- A. Git is sufficient for versioning AI services if everyone commits their changes
- B. The prompt template is a separate moving part that needs its own version record tied to each deployment
- C. The model provider made a silent update to the model endpoint
- D. Config files should be stored in the database, not on disk

**2. What is the correct way to compute the config_hash so it is stable and reproducible?**

- A. Hash the config dict using Python's built-in hash() function
- B. Use SHA-256 on the JSON string with keys sorted alphabetically
- C. Use the file modification timestamp of the config file
- D. Use a random UUID generated at registration time

**3. Why does the registry reject this and what should the engineer use instead?**

- A. The model ID format is wrong; it needs a version suffix like '-v1'
- B. Model aliases like 'claude-haiku-latest' can be silently updated by the provider, causing behavior changes you cannot trace; use the full pinned ID like 'claude-3-5-haiku-20241022'
- C. The manifest already has a model with 'haiku' in the name; only one per registry is allowed
- D. The registry only accepts OpenAI model IDs

**4. What is the fastest way to determine what model and prompt version were active 6 hours ago?**

- A. Check git blame on the config file for the last edit timestamp
- B. Ask the model provider which model version was serving traffic at that time
- C. Search the startup logs for '=== SERVICE STARTUP ===' entries before the incident started and read the manifest_id, model_id, and prompt_version
- D. Check the database for the last API response before the incident

**5. What is the most likely reason rollback did not fix the issue?**

- A. The rollback function has a bug that writes the wrong manifest ID
- B. The registry YAML file is read-only and the write failed silently
- C. rollback() only changes which manifest is marked as current in the file - the running service must restart to load the new active manifest
- D. The service cached the previous manifest in memory and must be rebuilt from source

**6. Is the colleague correct, and what does config_hash equality actually tell you?**

- A. Yes, equal config hashes mean identical behavior because all three components match
- B. No, config_hash only covers the service config dict (temperature, max_tokens, etc.). A different prompt_version means different prompts were used, which can significantly change model output
- C. Yes, but only if the model_id is also the same between the two deployments
- D. No, config_hash equality means the hash function collided and the configs are different

**7. What information should the CI pipeline inject into the manifest automatically?**

- A. The manifest_id should be the git commit SHA, deployed_by should be the CI system name, and deployed_at should be the pipeline start time
- B. The manifest_id should be a UUID generated by the CI system
- C. The CI pipeline should not create manifests; only human engineers should do so to ensure accuracy
- D. The CI pipeline should store the manifest in the database instead of a YAML file

**8. What is the best argument for storing only the hash rather than the full config?**

- A. The hash is more secure because it hides the config values from potential attackers
- B. Storing the full config would double the file size unnecessarily
- C. The hash keeps the manifest compact and diffable. When you need to inspect what changed, you look up the config file by its hash. The manifest is the index; the config file is the source of truth
- D. Hashes are required by the manifest spec and full configs are not supported

_Answers are in `checks.json` in the lesson directory._